# 04 — Creator leaderboard

Multi-metric scorecard per creator at the primary horizon (5d).

**How to read this:**
- `n_activated` matters more than any rate. Hit rates with N<30 are inferentially useless — Wilson CIs will tell you they overlap with chance.
- `mean_excess_return` is the headline number: creator return − SPY return over the same window. This controls for market regime.
- `hit_rate_lower_ci` is the more honest read of hit rate. If it's below 0.5, the creator does not statistically beat a coin flip yet.
- Recency contrast (`365d` vs `90d`) is the most informative single comparison: are they getting better or worse?

Run `tsr score` first to (re-)populate `creator_scorecards`.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sqlalchemy import select

from app.db import session_scope
from app.models import Creator, CreatorScorecard

with session_scope() as s:
    rows = s.execute(
        select(
            Creator.display_name.label('creator'),
            CreatorScorecard.window_label.label('window'),
            CreatorScorecard.horizon,
            CreatorScorecard.n_calls,
            CreatorScorecard.n_activated,
            CreatorScorecard.n_unique_tickers,
            CreatorScorecard.activation_rate,
            CreatorScorecard.hit_rate,
            CreatorScorecard.hit_rate_lower_ci,
            CreatorScorecard.hit_rate_upper_ci,
            CreatorScorecard.mean_return,
            CreatorScorecard.median_return,
            CreatorScorecard.mean_excess_return,
            CreatorScorecard.expectancy_unconditional,
            CreatorScorecard.sharpe_like,
            CreatorScorecard.mean_mae,
        )
        .join(CreatorScorecard, CreatorScorecard.creator_id == Creator.id)
    ).all()

df = pd.DataFrame(rows)
print(f'{len(df)} scorecards')
df.head()

## Primary view: 5d horizon, all-time

In [ ]:
primary = df[(df['horizon'] == '5d') & (df['window'] == 'all')].copy()
cols = ['creator', 'n_calls', 'n_activated', 'n_unique_tickers',
        'hit_rate', 'hit_rate_lower_ci', 'hit_rate_upper_ci',
        'mean_return', 'mean_excess_return', 'expectancy_unconditional',
        'sharpe_like', 'mean_mae']
primary[cols].sort_values('mean_excess_return', ascending=False).round(4)

## Recency drift (5d horizon)

Compare `all` vs `365d` vs `90d` for each creator. A creator who's recently improving should show `90d > 365d`; a creator decaying shows the opposite.

In [ ]:
h5 = df[df['horizon'] == '5d'].copy()
pivot = h5.pivot_table(
    index='creator',
    columns='window',
    values=['n_activated', 'mean_excess_return'],
)
pivot.round(4)

## All horizons for the top creator

In [ ]:
if not primary.empty:
    top = primary.sort_values('mean_excess_return', ascending=False).iloc[0]['creator']
    by_horizon = df[(df['creator'] == top) & (df['window'] == 'all')].sort_values('horizon')
    print(f'top creator: {top}')
    by_horizon[['horizon', 'n_calls', 'n_activated', 'hit_rate', 'mean_return', 'mean_excess_return']]
else:
    print('no scorecards')

## Honest reading

With current bootstrap N (single-digit calls per creator), nothing here is inferentially solid. The Wilson lower-CI bounds will all be wide enough to overlap with chance. The leaderboard is functional but **not yet meaningful**.

Re-evaluate when:
- The Whisper backfill has completed for the IP-blocked creators (currently only Trade Risk + Mancini have transcripts).
- At least 30 evaluated calls per creator have accumulated (target: 3–6 months of ingestion).
- The 21d horizon has caught up for the most recent calls (currently many are deferred via the `data_missing` retry path).